# Q8: Results

**Phase 9:** Results & Insights  
**Points: 3 points**

**Focus:** Generate final visualizations, create summary tables,
document key findings.

**Lecture Reference:** See **Lecture 11, Notebook 4**
(`11/demo/04_modeling_results.ipynb`), Phase 9 for examples of final
visualizations and results communication.


## Objective

Generate final visualizations, create summary tables, and document key
findings.

In [17]:
# Q8: Final Visualizations, Summary Table, Key Findings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Ensure output directory exists
os.makedirs("output", exist_ok=True)

# -------------------------
# Load Q7 outputs
# -------------------------
predictions = pd.read_csv("output/q7_predictions.csv")
feature_importance = pd.read_csv("output/q7_feature_importance.csv")

# Expect predictions to contain:
# 'actual', 'predicted_linear', 'predicted_random_forest', 'predicted_xgboost'
# Map expected prediction column names -> model friendly names
pred_to_model = {
    "predicted_linear": "Linear Regression",
    "predicted_random_forest": "Random Forest",
    "predicted_xgboost": "XGBoost"
}

# Ensure 'actual' exists
if "actual" not in predictions.columns:
    raise ValueError("predictions CSV must contain an 'actual' column.")

# -------------------------
# Compute metrics per model (safe if column missing)
# -------------------------
metrics = {}
for pred_col, model_name in pred_to_model.items():
    if pred_col in predictions.columns:
        y_true = predictions["actual"].values
        y_pred = predictions[pred_col].values
        r2 = r2_score(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        metrics[model_name] = {"R2": r2, "RMSE": rmse, "MAE": mae}
    else:
        metrics[model_name] = {"R2": np.nan, "RMSE": np.nan, "MAE": np.nan}

# -------------------------
# Build q8_summary.csv exactly with columns Metric,Linear Regression,Random Forest,XGBoost
# -------------------------
summary_df = pd.DataFrame({
    "Metric": ["R² Score", "RMSE", "MAE"],
    "Linear Regression": [metrics["Linear Regression"]["R2"],
                          metrics["Linear Regression"]["RMSE"],
                          metrics["Linear Regression"]["MAE"]],
    "Random Forest": [metrics["Random Forest"]["R2"],
                      metrics["Random Forest"]["RMSE"],
                      metrics["Random Forest"]["MAE"]],
    "XGBoost": [metrics["XGBoost"]["R2"],
                metrics["XGBoost"]["RMSE"],
                metrics["XGBoost"]["MAE"]],
})

# Save csv (no index)
summary_df.to_csv("output/q8_summary.csv", index=False)
print("Saved: output/q8_summary.csv")

# -------------------------
# Create visualizations (4-panel)
# -------------------------
plt.style.use("default")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: Model performance comparison (R², RMSE, MAE) as grouped bars
ax = axes[0, 0]
metrics_plot = summary_df.melt(id_vars="Metric", var_name="Model", value_name="Value")
# Plot grouped bars
sns.barplot(data=metrics_plot, x="Metric", y="Value", hue="Model", ax=ax)
ax.set_title("Model Performance Comparison (R², RMSE, MAE)")
ax.set_xlabel("")
ax.set_ylabel("Metric value")
ax.legend(title="Model", loc="best")

# Panel 2: Predictions vs Actual scatter (each model)
ax = axes[0, 1]
colors = {"Linear Regression": "tab:blue", "Random Forest": "tab:green", "XGBoost": "tab:orange"}
for pred_col, model_name in pred_to_model.items():
    if pred_col in predictions.columns:
        ax.scatter(predictions["actual"], predictions[pred_col], label=model_name, alpha=0.5, s=20)
# Perfect prediction line
min_val = predictions["actual"].min()
max_val = predictions["actual"].max()
ax.plot([min_val, max_val], [min_val, max_val], "k--", linewidth=1.5, label="Perfect")
ax.set_title("Predictions vs Actual")
ax.set_xlabel("Actual")
ax.set_ylabel("Predicted")
ax.legend()

# Panel 3: Feature importance (top N)
ax = axes[1, 0]
if feature_importance.shape[0] > 0:
    top_n = min(15, feature_importance.shape[0])
    top_features = feature_importance.sort_values("importance", ascending=False).head(top_n)
    sns.barplot(x="importance", y="feature", data=top_features, ax=ax)
    ax.set_title(f"Top {top_n} Feature Importances (tree-based model)")
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")
else:
    ax.text(0.5, 0.5, "No feature importance available", ha="center", va="center")
    ax.set_title("Feature Importance")
    ax.set_axis_off()

# Panel 4: Residuals plot (choose XGBoost if available, else first available)
ax = axes[1, 1]
chosen_pred_col = None
for col in ["predicted_xgboost", "predicted_random_forest", "predicted_linear"]:
    if col in predictions.columns:
        chosen_pred_col = col
        break

if chosen_pred_col is not None:
    resid = predictions["actual"] - predictions[chosen_pred_col]
    ax.scatter(predictions[chosen_pred_col], resid, alpha=0.5, s=20)
    ax.axhline(0, color="red", linestyle="--")
    model_label = pred_to_model.get(chosen_pred_col, chosen_pred_col)
    ax.set_title(f"Residuals vs Predicted ({model_label})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Residual (actual - predicted)")
else:
    ax.text(0.5, 0.5, "No model predictions available for residuals", ha="center", va="center")
    ax.set_axis_off()

plt.tight_layout()
plt.suptitle("Final Summary Visualizations (Q8)", fontsize=16, y=1.02)
plt.savefig("output/q8_final_visualizations.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: output/q8_final_visualizations.png")

# -------------------------
# Compose key findings (text)
# -------------------------
# Determine best model by R2 (ignore NaN)
best_model = None
best_r2 = -np.inf
for model_name, vals in metrics.items():
    r2v = vals["R2"]
    if not np.isnan(r2v) and r2v > best_r2:
        best_r2 = r2v
        best_model = model_name

# Top feature summary
top_features_list = []
if feature_importance.shape[0] > 0:
    fi_sorted = feature_importance.sort_values("importance", ascending=False)
    top_features_list = fi_sorted.head(3).apply(lambda r: f"{r['feature']} ({r['importance']:.3f})", axis=1).tolist()
    top3_sum = fi_sorted.head(3)["importance"].sum()
else:
    top3_sum = np.nan

with open("output/q8_key_findings.txt", "w") as f:
    f.write("KEY FINDINGS SUMMARY\n")
    f.write("===================\n\n")
    # Model performance
    f.write("MODEL PERFORMANCE:\n")
    if best_model is not None:
        f.write(f"- Best performing model: {best_model} (R² = {best_r2:.4f})\n")
    else:
        f.write("- Best performing model: None (no valid model R²)\n")
    # list metrics per model
    for model_name in ["Linear Regression", "Random Forest", "XGBoost"]:
        vals = metrics.get(model_name, {"R2": np.nan, "RMSE": np.nan, "MAE": np.nan})
        f.write(f"- {model_name}: R² = {vals['R2'] if not np.isnan(vals['R2']) else 'NaN'}, "
                f"RMSE = {vals['RMSE'] if not np.isnan(vals['RMSE']) else 'NaN'}, "
                f"MAE = {vals['MAE'] if not np.isnan(vals['MAE']) else 'NaN'}\n")
    f.write("\nFEATURE IMPORTANCE:\n")
    if top_features_list:
        f.write(f"- Top features: {', '.join(top_features_list)}\n")
        f.write(f"- Top 3 features account for {top3_sum*100:.1f}% of total importance\n")
    else:
        f.write("- No feature importance available\n")
    f.write("\nTEMPORAL PATTERNS:\n")
    f.write("- Daily (hour) and monthly cycles were used as temporal features and contribute to model performance (see feature importances).\n")
    f.write("\nDATA QUALITY:\n")
    f.write("- Dataset was cleaned in earlier steps (missing values handled, outliers capped via IQR method) as documented in Q2/Q3.\n")
    f.write(f"- Predictions file rows (test set size): {len(predictions)}\n")

print("Saved: output/q8_key_findings.txt")


Saved: output/q8_summary.csv
Saved: output/q8_final_visualizations.png
Saved: output/q8_key_findings.txt
